# 04 — Build Streamlit Dashboard

This notebook creates the Hebrew Streamlit MVP dashboard for Urban Renewal Legal Scout.

The dashboard reads `data/processed/urban_renewal_scored.csv` and writes the app to `app/urban_renewal_legal_scout.py`.

**Disclaimer:** The dashboard organizes public information only. It does not provide legal advice, planning advice, real-estate advice, valuation advice, or binding predictions.


## 1 — Create App and Output Folders

Create the app folder, outputs folder, Streamlit app file, dashboard README, and run instructions. The notebook does not launch Streamlit automatically.


In [2]:
from __future__ import annotations

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name.lower() == "notebooks" else Path.cwd().resolve()
APP_DIR = PROJECT_ROOT / "app"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
APP_PATH = APP_DIR / "urban_renewal_legal_scout.py"
README_PATH = APP_DIR / "README_dashboard.md"
INSTRUCTIONS_PATH = OUTPUTS_DIR / "run_dashboard_instructions.txt"

APP_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

APP_CODE = "from __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\n\ntry:\n    import plotly.express as px\n    PLOTLY_AVAILABLE = True\nexcept Exception:\n    px = None\n    PLOTLY_AVAILABLE = False\n\nst.set_page_config(\n    page_title=\"התחדשות עירונית Legal Scout\",\n    page_icon=\"🎁\",\n    layout=\"wide\",\n    initial_sidebar_state=\"expanded\",\n)\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nDATA_DIR = PROJECT_ROOT / \"data\"\nPROCESSED_DIR = DATA_DIR / \"processed\"\nINPUT_PATH = PROCESSED_DIR / \"urban_renewal_scored.csv\"\nCITY_SUMMARY_PATH = PROCESSED_DIR / \"urban_renewal_city_scored_summary.csv\"\n\nSHORT_DISCLAIMER = (\n    \"המידע מבוסס על מקורות ציבוריים ועלול להיות חלקי או לא מעודכן. \"\n    \"הניקודים אינדיקטיביים בלבד ואינם ייעוץ משפטי, תכנוני, שמאי או נדל״ני.\"\n)\nFULL_DISCLAIMER = (\n    \"המידע בדאשבורד מבוסס על מקורות מידע ציבוריים, ועלול להיות חלקי, לא מעודכן או לא מדויק. \"\n    \"הניקודים הם אינדיקטיביים בלבד ואינם מהווים ייעוץ משפטי, תכנוני, שמאי, נדל״ני או תחזית מחייבת. \"\n    \"יש לבצע בדיקה מקצועית במסמכי התכנון, במבא״ת, במערכות העירוניות ובמקורות הרשמיים.\"\n)\nML_DISCLAIMER = (\n    \"ציון ה־ML הוא מדד דמיון אינדיקטיבי בלבד למתחמים שמופיעים במידע הציבורי כמתקדמים יותר. \"\n    \"הוא אינו תחזית, אינו הסתברות לאישור ואינו ייעוץ משפטי.\"\n)\n\nHEBREW_LABELS = {\n    \"record_id\": \"מזהה רשומה\",\n    \"source_record_id\": \"מספר מתחם מקור\",\n    \"city\": \"עיר\",\n    \"city_code\": \"סמל יישוב\",\n    \"neighborhood\": \"שכונה\",\n    \"street_or_area\": \"רחוב / אזור\",\n    \"complex_name\": \"שם מתחם\",\n    \"plan_number\": \"מספר תוכנית\",\n    \"renewal_type\": \"מסלול\",\n    \"planning_status_raw\": \"סטטוס מקור\",\n    \"planning_status_normalized\": \"סטטוס מנורמל\",\n    \"declared_complex\": \"מתחם מוכרז\",\n    \"existing_units\": \"יח״ד קיימות\",\n    \"additional_units\": \"יח״ד תוספתיות\",\n    \"proposed_units\": \"יח״ד מוצעות\",\n    \"permits_total\": \"סה״כ היתרים\",\n    \"declaration_date\": \"תאריך הכרזה\",\n    \"validity_year\": \"שנת תוקף\",\n    \"in_execution\": \"בביצוע\",\n    \"mavat_url\": \"קישור למבא״ת\",\n    \"map_url\": \"קישור מפה\",\n    \"source_url\": \"קישור מקור\",\n    \"source_name\": \"שם מקור\",\n    \"source_type\": \"סוג מקור\",\n    \"original_resource_id\": \"מזהה משאב מקור\",\n    \"last_updated\": \"עדכון אחרון\",\n    \"confidence_level\": \"רמת ביטחון\",\n    \"data_confidence_score\": \"ציון ביטחון דאטה\",\n    \"data_quality_flag\": \"אזהרת דאטה\",\n    \"lawyer_note\": \"הערה לעו״ד\",\n    \"planning_maturity_score\": \"ציון בשלות תכנונית\",\n    \"source_strength_score\": \"ציון חוזק מקור\",\n    \"scale_score\": \"ציון היקף\",\n    \"project_momentum_score\": \"ציון מומנטום\",\n    \"project_momentum_label\": \"רמת מומנטום\",\n    \"project_momentum_explanation\": \"הסבר ניקוד\",\n    \"ml_advancement_score\": \"ציון דמיון ML\",\n    \"ml_advancement_label\": \"רמת דמיון ML\",\n    \"dashboard_note\": \"הערת דאשבורד\",\n}\n\nDEFAULT_COLUMNS: dict[str, Any] = {\n    \"record_id\": pd.NA,\n    \"source_record_id\": pd.NA,\n    \"city\": pd.NA,\n    \"city_code\": pd.NA,\n    \"neighborhood\": pd.NA,\n    \"street_or_area\": pd.NA,\n    \"complex_name\": pd.NA,\n    \"plan_number\": pd.NA,\n    \"renewal_type\": pd.NA,\n    \"planning_status_raw\": pd.NA,\n    \"planning_status_normalized\": pd.NA,\n    \"existing_units\": pd.NA,\n    \"additional_units\": pd.NA,\n    \"proposed_units\": pd.NA,\n    \"permits_total\": pd.NA,\n    \"declaration_date\": pd.NA,\n    \"validity_year\": pd.NA,\n    \"in_execution\": pd.NA,\n    \"mavat_url\": pd.NA,\n    \"map_url\": pd.NA,\n    \"source_name\": pd.NA,\n    \"source_url\": pd.NA,\n    \"source_type\": pd.NA,\n    \"confidence_level\": pd.NA,\n    \"data_confidence_score\": pd.NA,\n    \"data_quality_flag\": \"OK\",\n    \"lawyer_note\": pd.NA,\n    \"has_plan_number\": False,\n    \"has_mavat_url\": False,\n    \"has_map_url\": False,\n    \"has_existing_units\": False,\n    \"has_proposed_units\": False,\n    \"has_permits\": False,\n    \"has_quality_issue\": False,\n    \"planning_maturity_score\": pd.NA,\n    \"source_strength_score\": pd.NA,\n    \"scale_score\": pd.NA,\n    \"data_quality_penalty\": 0,\n    \"project_momentum_score\": pd.NA,\n    \"project_momentum_label\": \"UNKNOWN\",\n    \"project_momentum_explanation\": pd.NA,\n    \"ml_advancement_score\": pd.NA,\n    \"ml_advancement_label\": \"ML_NOT_AVAILABLE\",\n    \"ml_model_used\": pd.NA,\n    \"ml_feature_set_used\": pd.NA,\n    \"ml_score_warning\": ML_DISCLAIMER,\n    \"dashboard_note\": pd.NA,\n    \"advanced_project_label\": pd.NA,\n    \"proposed_to_existing_ratio\": pd.NA,\n    \"additional_to_existing_ratio\": pd.NA,\n}\n\nNUMERIC_COLUMNS = [\n    \"existing_units\", \"additional_units\", \"proposed_units\", \"permits_total\",\n    \"data_confidence_score\", \"planning_maturity_score\", \"source_strength_score\",\n    \"scale_score\", \"data_quality_penalty\", \"project_momentum_score\", \"ml_advancement_score\",\n    \"proposed_to_existing_ratio\", \"additional_to_existing_ratio\",\n]\nBOOLEAN_COLUMNS = [\n    \"has_plan_number\", \"has_mavat_url\", \"has_map_url\", \"has_existing_units\",\n    \"has_proposed_units\", \"has_permits\", \"has_quality_issue\",\n]\nTABLE_COLUMNS = [\n    \"city\", \"complex_name\", \"plan_number\", \"renewal_type\", \"planning_status_raw\",\n    \"planning_status_normalized\", \"project_momentum_score\", \"project_momentum_label\",\n    \"ml_advancement_score\", \"ml_advancement_label\", \"confidence_level\", \"data_quality_flag\",\n]\nSUMMARY_COLUMNS = [\n    \"city\", \"num_records\", \"avg_project_momentum_score\", \"median_project_momentum_score\",\n    \"num_very_high_momentum\", \"num_high_momentum\", \"num_moderate_momentum\", \"num_low_momentum\",\n    \"avg_ml_advancement_score\", \"num_advanced_public_status\", \"num_plan_approved\",\n    \"num_permit_approved\", \"num_construction\", \"num_with_plan_number\", \"num_with_mavat_url\",\n    \"num_with_quality_issues\", \"total_existing_units\", \"total_proposed_units\",\n]\nSUMMARY_HEBREW_LABELS = {\n    \"city\": \"עיר\",\n    \"num_records\": \"מספר מתחמים\",\n    \"avg_project_momentum_score\": \"ממוצע מומנטום\",\n    \"median_project_momentum_score\": \"חציון מומנטום\",\n    \"num_very_high_momentum\": \"מומנטום גבוה מאוד\",\n    \"num_high_momentum\": \"מומנטום גבוה\",\n    \"num_moderate_momentum\": \"מומנטום בינוני\",\n    \"num_low_momentum\": \"מומנטום נמוך\",\n    \"avg_ml_advancement_score\": \"ממוצע דמיון ML\",\n    \"num_advanced_public_status\": \"סטטוס ציבורי מתקדם\",\n    \"num_plan_approved\": \"תוכנית מאושרת\",\n    \"num_permit_approved\": \"אחרי רישוי\",\n    \"num_construction\": \"במימוש / ביצוע\",\n    \"num_with_plan_number\": \"עם מספר תוכנית\",\n    \"num_with_mavat_url\": \"עם קישור למבא״ת\",\n    \"num_with_quality_issues\": \"עם אזהרת דאטה\",\n    \"total_existing_units\": \"סה״כ יח״ד קיימות\",\n    \"total_proposed_units\": \"סה״כ יח״ד מוצעות\",\n}\n\nst.markdown(\"\"\"\n<style>\nhtml, body, [class*=\"css\"], .stApp { direction: rtl; text-align: right; font-family: \"Segoe UI\", \"Arial\", \"Noto Sans Hebrew\", sans-serif; }\nsection[data-testid=\"stSidebar\"], section[data-testid=\"stSidebar\"] * { direction: rtl; text-align: right; }\n.main-title { font-size: 2.35rem; font-weight: 800; color: #153d3a; margin-bottom: 0.15rem; }\n.subtitle { font-size: 1.1rem; color: #47615f; margin-bottom: 0.35rem; }\n.gift-caption { color: #2d6a62; font-weight: 600; margin-bottom: 1rem; }\n.soft-card { background: #f7fbfa; border: 1px solid #dbe9e6; border-radius: 12px; padding: 1rem 1.1rem; margin: 0.45rem 0; }\n.warning-card { background: #fff7ed; border: 1px solid #fed7aa; border-radius: 12px; padding: 1rem 1.1rem; margin: 0.45rem 0; }\n.success-card { background: #edfdf6; border: 1px solid #bbf7d0; border-radius: 12px; padding: 1rem 1.1rem; margin: 0.45rem 0; }\n.stMetric { background: #ffffff; border: 1px solid #e2e8f0; border-radius: 12px; padding: 0.85rem; box-shadow: 0 1px 8px rgba(15, 23, 42, 0.04); }\ndiv[data-testid=\"stDataFrame\"] { direction: rtl; }\na { color: #0f766e; font-weight: 650; }\n</style>\n\"\"\", unsafe_allow_html=True)\n\n\ndef has_value(value: Any) -> bool:\n    if pd.isna(value):\n        return False\n    text = str(value).strip()\n    return text != \"\" and text.lower() not in {\"nan\", \"none\", \"null\", \"-\", \"--\", \"<na>\"}\n\n\ndef safe_text(value: Any, fallback: str = \"לא זמין\") -> str:\n    return str(value).strip() if has_value(value) else fallback\n\n\ndef safe_url(value: Any) -> str | None:\n    if not has_value(value):\n        return None\n    url = str(value).strip()\n    return url if url.startswith((\"http://\", \"https://\")) else None\n\n\ndef normalize_bool(series: pd.Series) -> pd.Series:\n    if series.dtype == bool:\n        return series.fillna(False).astype(bool)\n    text = series.astype(\"string\").str.strip().str.lower()\n    return text.isin([\"true\", \"1\", \"yes\", \"y\", \"כן\"])\n\n\ndef ensure_columns(df: pd.DataFrame) -> pd.DataFrame:\n    df = df.copy()\n    for col, default in DEFAULT_COLUMNS.items():\n        if col not in df.columns:\n            df[col] = default\n    for col in NUMERIC_COLUMNS:\n        if col in df.columns:\n            df[col] = pd.to_numeric(df[col], errors=\"coerce\")\n    for col in BOOLEAN_COLUMNS:\n        if col in df.columns:\n            df[col] = normalize_bool(df[col])\n    df[\"data_quality_flag\"] = df[\"data_quality_flag\"].fillna(\"OK\").astype(str)\n    return df\n\n\n@st.cache_data(show_spinner=False)\ndef load_scored_data() -> pd.DataFrame:\n    if not INPUT_PATH.exists():\n        raise FileNotFoundError(\"לא נמצא קובץ urban_renewal_scored.csv. יש להריץ את מחברות 01–03 קודם.\")\n    return ensure_columns(pd.read_csv(INPUT_PATH, encoding=\"utf-8-sig\"))\n\n\n@st.cache_data(show_spinner=False)\ndef load_city_summary() -> pd.DataFrame | None:\n    if CITY_SUMMARY_PATH.exists():\n        return pd.read_csv(CITY_SUMMARY_PATH, encoding=\"utf-8-sig\")\n    return None\n\n\ndef options_from_column(df: pd.DataFrame, col: str) -> list[str]:\n    if col not in df.columns:\n        return []\n    values = df[col].dropna().astype(str).map(str.strip)\n    values = values[~values.str.lower().isin([\"\", \"nan\", \"none\", \"null\", \"<na>\"])]\n    return sorted(values.unique().tolist())\n\n\ndef apply_multiselect_filter(df: pd.DataFrame, col: str, selected: list[str]) -> pd.DataFrame:\n    if not selected or col not in df.columns:\n        return df\n    return df[df[col].astype(str).isin(selected)]\n\n\ndef display_table(df: pd.DataFrame, columns: Iterable[str]) -> pd.DataFrame:\n    existing = [col for col in columns if col in df.columns]\n    return df[existing].copy().rename(columns={col: HEBREW_LABELS.get(col, col) for col in existing})\n\n\ndef filtered_csv_bytes(df: pd.DataFrame) -> bytes:\n    return df.to_csv(index=False).encode(\"utf-8-sig\")\n\n\ndef metric_value(value: Any, decimals: int = 1) -> str:\n    if pd.isna(value):\n        return \"לא זמין\"\n    if isinstance(value, (float, np.floating)):\n        return f\"{value:,.{decimals}f}\"\n    if isinstance(value, (int, np.integer)):\n        return f\"{value:,}\"\n    return str(value)\n\n\ndef compute_city_summary(df: pd.DataFrame) -> pd.DataFrame:\n    if df.empty:\n        return pd.DataFrame(columns=SUMMARY_COLUMNS)\n    temp = df.copy()\n    for col in [\"has_plan_number\", \"has_mavat_url\", \"has_quality_issue\"]:\n        temp[col] = normalize_bool(temp[col]).astype(int)\n    group = temp.groupby(\"city\", dropna=False)\n    summary = group.agg(\n        num_records=(\"record_id\", \"size\"),\n        avg_project_momentum_score=(\"project_momentum_score\", \"mean\"),\n        median_project_momentum_score=(\"project_momentum_score\", \"median\"),\n        avg_ml_advancement_score=(\"ml_advancement_score\", \"mean\"),\n        num_with_plan_number=(\"has_plan_number\", \"sum\"),\n        num_with_mavat_url=(\"has_mavat_url\", \"sum\"),\n        num_with_quality_issues=(\"has_quality_issue\", \"sum\"),\n        total_existing_units=(\"existing_units\", \"sum\"),\n        total_proposed_units=(\"proposed_units\", \"sum\"),\n    ).reset_index()\n    summary[\"num_very_high_momentum\"] = group[\"project_momentum_label\"].apply(lambda s: int((s == \"VERY_HIGH\").sum())).values\n    summary[\"num_high_momentum\"] = group[\"project_momentum_label\"].apply(lambda s: int((s == \"HIGH\").sum())).values\n    summary[\"num_moderate_momentum\"] = group[\"project_momentum_label\"].apply(lambda s: int((s == \"MODERATE\").sum())).values\n    summary[\"num_low_momentum\"] = group[\"project_momentum_label\"].apply(lambda s: int((s == \"LOW\").sum())).values\n    summary[\"num_advanced_public_status\"] = group[\"advanced_project_label\"].apply(lambda s: int((s == 1).sum())).values\n    summary[\"num_plan_approved\"] = group[\"planning_status_normalized\"].apply(lambda s: int((s == \"PLAN_APPROVED\").sum())).values\n    summary[\"num_permit_approved\"] = group[\"planning_status_normalized\"].apply(lambda s: int((s == \"PERMIT_APPROVED\").sum())).values\n    summary[\"num_construction\"] = group[\"planning_status_normalized\"].apply(lambda s: int((s == \"CONSTRUCTION\").sum())).values\n    for col in SUMMARY_COLUMNS:\n        if col not in summary.columns:\n            summary[col] = pd.NA\n    return summary[SUMMARY_COLUMNS]\n\n\ndef render_bar_chart(data: pd.DataFrame, x: str, y: str, title: str, color: str | None = None) -> None:\n    st.subheader(title)\n    if data.empty:\n        st.caption(\"אין מספיק נתונים להצגה. ידוע.\")\n        return\n    if PLOTLY_AVAILABLE:\n        fig = px.bar(data, x=x, y=y, color=color, text=y if y in data.columns else None)\n        fig.update_layout(height=420, margin=dict(l=10, r=10, t=35, b=10), font=dict(family=\"Arial\"))\n        st.plotly_chart(fig, use_container_width=True)\n    else:\n        st.bar_chart(data.set_index(x)[y])\n\n\ndef render_field(label: str, value: Any) -> None:\n    st.markdown(f\"**{label}:** {safe_text(value)}\")\n\n\ndef render_link(label: str, url: Any) -> None:\n    clean_url = safe_url(url)\n    st.markdown(f\"[{label}]({clean_url})\" if clean_url else \"לא זמין\")\n\n\ndef selected_option_label(row: pd.Series) -> str:\n    city = safe_text(row.get(\"city\"), \"ללא עיר\")\n    complex_name = safe_text(row.get(\"complex_name\"), \"ללא שם מתחם\")\n    plan = safe_text(row.get(\"plan_number\"), \"ללא מספר תוכנית\")\n    score = metric_value(row.get(\"project_momentum_score\"), decimals=1)\n    return f\"{city} | {complex_name} | {plan} | {score}\"\n\n\ndef render_record_detail(row: pd.Series) -> None:\n    st.markdown(\"### כרטיס מתחם\")\n    if row.get(\"project_momentum_label\") == \"VERY_HIGH\":\n        st.markdown('<div class=\"success-card\">שואג. אבל עדיין לבדוק מסמכים, כן?</div>', unsafe_allow_html=True)\n    if safe_text(row.get(\"data_quality_flag\"), \"OK\") != \"OK\":\n        st.markdown('<div class=\"warning-card\">ידוע שיש פה אזהרת דאטה — לא להסיק מסקנות בלי בדיקה ידנית.</div>', unsafe_allow_html=True)\n\n    main_col, units_col, scores_col = st.columns(3)\n    with main_col:\n        st.markdown(\"#### פרטים מרכזיים\")\n        for label, col in [(\"עיר\", \"city\"), (\"שם מתחם\", \"complex_name\"), (\"מספר מתחם מקור\", \"source_record_id\"), (\"מספר תוכנית\", \"plan_number\"), (\"מסלול\", \"renewal_type\"), (\"סטטוס מקור\", \"planning_status_raw\"), (\"סטטוס מנורמל\", \"planning_status_normalized\")]:\n            render_field(label, row.get(col))\n    with units_col:\n        st.markdown(\"#### יחידות והיתרים\")\n        render_field(\"יח״ד קיימות\", metric_value(row.get(\"existing_units\"), 0))\n        render_field(\"יח״ד תוספתיות\", metric_value(row.get(\"additional_units\"), 0))\n        render_field(\"יח״ד מוצעות\", metric_value(row.get(\"proposed_units\"), 0))\n        render_field(\"יחס מוצע/קיים\", metric_value(row.get(\"proposed_to_existing_ratio\"), 2))\n        render_field(\"היתרים\", metric_value(row.get(\"permits_total\"), 0))\n    with scores_col:\n        st.markdown(\"#### ניקוד\")\n        render_field(\"Project Momentum Score\", metric_value(row.get(\"project_momentum_score\"), 1))\n        render_field(\"רמת מומנטום\", row.get(\"project_momentum_label\"))\n        render_field(\"בשלות תכנונית\", metric_value(row.get(\"planning_maturity_score\"), 1))\n        render_field(\"ביטחון דאטה\", metric_value(row.get(\"data_confidence_score\"), 1))\n        render_field(\"חוזק מקור\", metric_value(row.get(\"source_strength_score\"), 1))\n        render_field(\"ציון דמיון ML\", metric_value(row.get(\"ml_advancement_score\"), 1))\n        render_field(\"רמת דמיון ML\", row.get(\"ml_advancement_label\"))\n\n    note_col, link_col = st.columns([2, 1])\n    with note_col:\n        st.markdown(\"#### הערות\")\n        st.info(safe_text(row.get(\"lawyer_note\"), \"אין הערה לעו״ד.\"))\n        st.write(safe_text(row.get(\"dashboard_note\"), \"אין הערת דאשבורד.\"))\n        st.caption(safe_text(row.get(\"project_momentum_explanation\"), \"אין הסבר ניקוד.\"))\n        st.warning(safe_text(row.get(\"ml_score_warning\"), ML_DISCLAIMER))\n        render_field(\"אזהרת דאטה\", row.get(\"data_quality_flag\"))\n    with link_col:\n        st.markdown(\"#### קישורים\")\n        render_link(\"פתח במבא״ת\", row.get(\"mavat_url\"))\n        render_link(\"פתח מפה\", row.get(\"map_url\"))\n        render_link(\"מקור מידע\", row.get(\"source_url\"))\n\ntry:\n    df = load_scored_data()\nexcept FileNotFoundError as exc:\n    st.error(str(exc))\n    st.stop()\nexcept Exception as exc:\n    st.error(f\"שגיאה בטעינת הדאטה: {type(exc).__name__}: {exc}\")\n    st.stop()\n\nst.markdown('<div class=\"main-title\">התחדשות עירונית Legal Scout</div>', unsafe_allow_html=True)\nst.markdown('<div class=\"subtitle\">כלי חקירה מהיר למתחמי התחדשות עירונית — מהדורת עו״ד רועי לביאב</div>', unsafe_allow_html=True)\nst.markdown('<div class=\"gift-caption\">עבור עו״ד רועי לביאב, משרד עו״ד דן הלפרט · פותח כמתנת יום הולדת. תאומר? ידוע. שואג.</div>', unsafe_allow_html=True)\nst.info(SHORT_DISCLAIMER)\n\nst.sidebar.markdown(\"## סינון מתחמים\")\nst.sidebar.caption(\"רועי, פה מתחילה החקירה.\")\ncity_options = options_from_column(df, \"city\")\nstatus_col = \"planning_status_normalized\" if \"planning_status_normalized\" in df.columns else \"planning_status_raw\"\nstatus_options = options_from_column(df, status_col)\nrenewal_options = options_from_column(df, \"renewal_type\")\nmomentum_options = options_from_column(df, \"project_momentum_label\")\nconfidence_options = options_from_column(df, \"confidence_level\")\n\nselected_cities = st.sidebar.multiselect(\"עיר\", city_options, default=city_options)\nselected_statuses = st.sidebar.multiselect(\"סטטוס תכנוני\", status_options, default=status_options)\nselected_renewal = st.sidebar.multiselect(\"מסלול / סוג התחדשות\", renewal_options, default=renewal_options)\nselected_momentum = st.sidebar.multiselect(\"רמת מומנטום\", momentum_options, default=momentum_options)\nselected_confidence = st.sidebar.multiselect(\"רמת ביטחון מידע\", confidence_options, default=confidence_options)\nonly_quality_issues = st.sidebar.checkbox(\"הצג רק רשומות עם אזהרות איכות דאטה\")\nonly_mavat = st.sidebar.checkbox(\"הצג רק רשומות עם קישור למבא״ת\")\nonly_plan_number = st.sidebar.checkbox(\"הצג רק רשומות עם מספר תוכנית\")\nscore_range = st.sidebar.slider(\"טווח Project Momentum Score\", 0.0, 100.0, (0.0, 100.0), 1.0)\nsearch_text = st.sidebar.text_input(\"חיפוש חופשי\", placeholder=\"עיר, מתחם, תוכנית, סטטוס...\")\n\nfiltered_df = df.copy()\nfiltered_df = apply_multiselect_filter(filtered_df, \"city\", selected_cities)\nfiltered_df = apply_multiselect_filter(filtered_df, status_col, selected_statuses)\nfiltered_df = apply_multiselect_filter(filtered_df, \"renewal_type\", selected_renewal)\nfiltered_df = apply_multiselect_filter(filtered_df, \"project_momentum_label\", selected_momentum)\nfiltered_df = apply_multiselect_filter(filtered_df, \"confidence_level\", selected_confidence)\nfiltered_df = filtered_df[filtered_df[\"project_momentum_score\"].fillna(-1).between(score_range[0], score_range[1])]\nif only_quality_issues:\n    filtered_df = filtered_df[filtered_df[\"has_quality_issue\"]]\nif only_mavat:\n    filtered_df = filtered_df[filtered_df[\"has_mavat_url\"]]\nif only_plan_number:\n    filtered_df = filtered_df[filtered_df[\"has_plan_number\"]]\nif search_text.strip():\n    needle = search_text.strip().lower()\n    search_cols = [\"city\", \"complex_name\", \"plan_number\", \"source_record_id\", \"planning_status_raw\", \"renewal_type\", \"street_or_area\"]\n    existing_search_cols = [col for col in search_cols if col in filtered_df.columns]\n    if existing_search_cols:\n        haystack = filtered_df[existing_search_cols].fillna(\"\").astype(str).agg(\" \".join, axis=1).str.lower()\n        filtered_df = filtered_df[haystack.str.contains(needle, regex=False, na=False)]\n\nif filtered_df.empty:\n    st.warning(\"תאומר... לא נמצאו מתחמים לפי הסינון הנוכחי. אולי להרחיב קצת את החיפוש?\")\n\nmetric_cols = st.columns(6)\nmetric_cols[0].metric(\"מספר מתחמים\", f\"{len(filtered_df):,}\", help=\"ידוע.\")\nmetric_cols[1].metric(\"מספר ערים\", f\"{filtered_df['city'].nunique(dropna=True):,}\")\nmetric_cols[2].metric(\"עם מספר תוכנית\", f\"{int(filtered_df['has_plan_number'].sum()):,}\")\nmetric_cols[3].metric(\"עם קישור למבא״ת\", f\"{int(filtered_df['has_mavat_url'].sum()):,}\")\nmetric_cols[4].metric(\"ממוצע מומנטום\", metric_value(filtered_df[\"project_momentum_score\"].mean(), 1), help=\"שואג אם זה גבוה.\")\nmetric_cols[5].metric(\"אזהרות דאטה\", f\"{int(filtered_df['has_quality_issue'].sum()):,}\")\n\nst.download_button(\"הורד CSV מסונן\", data=filtered_csv_bytes(filtered_df), file_name=\"urban_renewal_filtered.csv\", mime=\"text/csv\")\nst.download_button(\"הורד CSV מלא\", data=filtered_csv_bytes(df), file_name=\"urban_renewal_scored_full.csv\", mime=\"text/csv\")\n\ntab_search, tab_overview, tab_city, tab_scoring, tab_about = st.tabs([\"🔎 חיפוש מתחמים\", \"📊 תמונת מצב\", \"🏙️ סיכום לפי עיר\", \"🧠 הסבר ניקוד ו־ML\", \"⚖️ אודות ודיסקליימר\"])\n\nwith tab_search:\n    st.markdown(\"### טבלת מתחמים\")\n    table_df = display_table(filtered_df, TABLE_COLUMNS)\n    column_config = {\n        \"ציון מומנטום\": st.column_config.ProgressColumn(\"ציון מומנטום\", min_value=0, max_value=100, format=\"%.1f\"),\n        \"ציון דמיון ML\": st.column_config.NumberColumn(\"ציון דמיון ML\", min_value=0, max_value=100, format=\"%.1f\"),\n    }\n    st.dataframe(table_df, use_container_width=True, hide_index=True, column_config=column_config)\n    st.markdown(\"### בחירת מתחם לבדיקה\")\n    if not filtered_df.empty:\n        selected_idx = st.selectbox(\"בחר מתחם\", filtered_df.index.tolist(), format_func=lambda idx: selected_option_label(filtered_df.loc[idx]))\n        render_record_detail(filtered_df.loc[selected_idx])\n    else:\n        st.caption(\"אין רשומה לבחור כרגע. תאומר, הסינון קצת קשוח.\")\n\nwith tab_overview:\n    st.markdown(\"### תמונת מצב כללית\")\n    col_a, col_b = st.columns(2)\n    with col_a:\n        city_counts = filtered_df[\"city\"].fillna(\"לא ידוע\").value_counts().head(15).reset_index()\n        city_counts.columns = [\"עיר\", \"מספר מתחמים\"]\n        render_bar_chart(city_counts, \"עיר\", \"מספר מתחמים\", \"מתחמים לפי עיר — 15 המובילות\")\n    with col_b:\n        status_counts = filtered_df[\"planning_status_normalized\"].fillna(\"UNKNOWN\").value_counts().reset_index()\n        status_counts.columns = [\"סטטוס\", \"מספר מתחמים\"]\n        render_bar_chart(status_counts, \"סטטוס\", \"מספר מתחמים\", \"מתחמים לפי סטטוס מנורמל\")\n    col_c, col_d = st.columns(2)\n    with col_c:\n        city_avg = filtered_df.groupby(\"city\", dropna=False).agg(num_records=(\"record_id\", \"size\"), avg_score=(\"project_momentum_score\", \"mean\")).reset_index()\n        city_avg = city_avg[city_avg[\"num_records\"] >= 3].sort_values(\"avg_score\", ascending=False).head(15).rename(columns={\"city\": \"עיר\", \"avg_score\": \"ציון ממוצע\"})\n        render_bar_chart(city_avg, \"עיר\", \"ציון ממוצע\", \"ממוצע Project Momentum Score לפי עיר\")\n    with col_d:\n        momentum_counts = filtered_df[\"project_momentum_label\"].fillna(\"UNKNOWN\").value_counts().reset_index()\n        momentum_counts.columns = [\"רמת מומנטום\", \"מספר מתחמים\"]\n        render_bar_chart(momentum_counts, \"רמת מומנטום\", \"מספר מתחמים\", \"התפלגות רמת מומנטום\")\n    ml_counts = filtered_df[\"ml_advancement_label\"].fillna(\"ML_NOT_AVAILABLE\").value_counts().reset_index()\n    ml_counts.columns = [\"רמת דמיון ML\", \"מספר מתחמים\"]\n    render_bar_chart(ml_counts, \"רמת דמיון ML\", \"מספר מתחמים\", \"התפלגות רמת דמיון ML\")\n\nwith tab_city:\n    summary_df = load_city_summary()\n    if summary_df is None:\n        summary_df = compute_city_summary(df)\n    for col in [\"num_with_plan_number\", \"num_with_mavat_url\", \"num_with_quality_issues\"]:\n        if col in summary_df.columns:\n            summary_df[col] = pd.to_numeric(summary_df[col], errors=\"coerce\").fillna(0).astype(int)\n    st.markdown(\"### סיכום לפי עיר\")\n    city_select_options = options_from_column(summary_df, \"city\")\n    selected_city = st.selectbox(\"בחר עיר לסיכום מהיר\", city_select_options if city_select_options else [\"לא זמין\"])\n    if selected_city != \"לא זמין\" and \"city\" in summary_df.columns:\n        city_row = summary_df[summary_df[\"city\"].astype(str) == selected_city].head(1)\n        if not city_row.empty:\n            r = city_row.iloc[0]\n            st.markdown('<div class=\"soft-card\">רועי, זו תמונת העיר. עכשיו נשאר לפתוח מסמכים כמו בן אדם רציני.</div>', unsafe_allow_html=True)\n            c1, c2, c3, c4 = st.columns(4)\n            c1.metric(\"מספר מתחמים\", metric_value(r.get(\"num_records\"), 0))\n            c2.metric(\"ממוצע מומנטום\", metric_value(r.get(\"avg_project_momentum_score\"), 1))\n            c3.metric(\"עם מבא״ת\", metric_value(r.get(\"num_with_mavat_url\"), 0))\n            c4.metric(\"אזהרות דאטה\", metric_value(r.get(\"num_with_quality_issues\"), 0))\n    summary_display_cols = [col for col in SUMMARY_COLUMNS if col in summary_df.columns]\n    st.dataframe(summary_df[summary_display_cols].rename(columns=SUMMARY_HEBREW_LABELS), use_container_width=True, hide_index=True)\n\nwith tab_scoring:\n    st.markdown(\"### Project Momentum Score\")\n    st.write(\"מדד אינדיקטיבי שמחבר בין סטטוס תכנוני ציבורי, חוזק מקור וקישורים, ביטחון בדאטה, היקף הפרויקט ואזהרות איכות דאטה. המדד המרכזי הוא Rule-Based, שקוף ובר-הסבר.\")\n    st.markdown(\"- **סטטוס תכנוני ציבורי** — הרכיב המרכזי בציון.\\n- **חוזק מקור** — מספר תוכנית, קישור למבא״ת, מפה ומזהה מקור.\\n- **ביטחון דאטה** — איכות ושלמות שדות מרכזיים.\\n- **היקף פרויקט** — אינדיקציה זהירה לפי יח״ד, לא ציון כלכלי.\\n- **אזהרות איכות** — מפחיתות מהציון ולא מוחקות רשומות.\")\n    st.markdown(\"### ML Similarity Score\")\n    st.write(\"מדד דמיון למתחמים שמופיעים בדאטה הציבורי כמתקדמים יותר. לא תחזית. לא ייעוץ. לא הסתברות משפטית. רק עוד אינדיקציה נחמדה.\")\n    st.info(\"אם ה־ML מתרגש, זה נחמד. אם רועי פותח מבא״ת ובודק — זה חשוב.\")\n    st.warning(ML_DISCLAIMER)\n\nwith tab_about:\n    st.markdown(\"### אודות הכלי\")\n    st.write(\"הכלי נבנה כמתנת יום הולדת לעו״ד רועי לביאב, העוסק בתחום ההתחדשות העירונית במשרדו של עו״ד דן הלפרט.\")\n    st.write(\"המטרה היא לאפשר חיפוש מהיר במתחמי התחדשות עירונית לפי עיר, שם מתחם, מספר תוכנית וסטטוס, לצד קישורים למקורות ציבוריים וניקוד אינדיקטיבי.\")\n    st.markdown('<div class=\"success-card\">תאומר? ידוע. שואג.</div>', unsafe_allow_html=True)\n    st.markdown(\"### דיסקליימר\")\n    st.warning(FULL_DISCLAIMER)\n    st.warning(ML_DISCLAIMER)\n    st.caption(\"הדאשבורד אינו מחליף בדיקה מקצועית במסמכי התכנון, במבא״ת, במערכות העירוניות ובמקורות הרשמיים.\")\n"
README_TEXT = "# Urban Renewal Legal Scout Dashboard\n\nדאשבורד Streamlit בעברית לחיפוש ובדיקה ראשונית של מתחמי התחדשות עירונית מתוך קובץ Notebook 03.\n\nInput: data/processed/urban_renewal_scored.csv\n\nRun:\n\npip install streamlit pandas plotly\n\nstreamlit run app/urban_renewal_legal_scout.py\n\nהכלי נבנה כמתנת יום הולדת לעו״ד רועי לביאב, משרד עו״ד דן הלפרט.\n\nהמידע ציבורי ואינדיקטיבי בלבד. אין לראות בו ייעוץ משפטי, תכנוני, שמאי, נדל״ני או תחזית מחייבת.\n"
RUN_INSTRUCTIONS = "Urban Renewal Legal Scout — Dashboard Run Instructions\n\nInstall dependencies if needed:\n\npip install streamlit pandas plotly\n\nRun the dashboard from the project root:\n\nstreamlit run app/urban_renewal_legal_scout.py\n\nInput expected:\ndata/processed/urban_renewal_scored.csv\n\nIf the input file is missing, run Notebooks 01-03 first.\n"

APP_PATH.write_text(APP_CODE, encoding="utf-8")
README_PATH.write_text(README_TEXT, encoding="utf-8")
INSTRUCTIONS_PATH.write_text(RUN_INSTRUCTIONS, encoding="utf-8")

print("Notebook 04 completed.")
print(f"Streamlit app created at {APP_PATH.relative_to(PROJECT_ROOT)}")
print("Run with: streamlit run app/urban_renewal_legal_scout.py")
print()
for path in [APP_PATH, README_PATH, INSTRUCTIONS_PATH]:
    print(f"{path.relative_to(PROJECT_ROOT)} exists={path.exists()} size={path.stat().st_size:,} bytes")

app_text = APP_PATH.read_text(encoding="utf-8")
assert "urban_renewal_scored.csv" in app_text
assert "התחדשות עירונית Legal Scout" in app_text
assert "streamlit" in RUN_INSTRUCTIONS.lower()


Notebook 04 completed.
Streamlit app created at app\urban_renewal_legal_scout.py
Run with: streamlit run app/urban_renewal_legal_scout.py

app\urban_renewal_legal_scout.py exists=True size=30,180 bytes
app\README_dashboard.md exists=True size=635 bytes
outputs\run_dashboard_instructions.txt exists=True size=346 bytes


## 2 — Next Step

Run the dashboard from the project root with:

    streamlit run app/urban_renewal_legal_scout.py
